In [3]:
import pandas as pd
import pickle
import os
import torch
import torch.nn as nn
from TorchCRF import CRF
from tqdm.auto import tqdm
from transformers import AutoTokenizer
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_converter import bio_to_indices

MODELS_DIR_V1 = "../../models/iteration-1/"
SUBMISSIONS_DIR = "../../submissions/iteration-1/"
ARTEFACTS_PATH_V1 = os.path.join(MODELS_DIR_V1, "artefacts_v1.pkl")
MODEL_PATH_V1 = os.path.join(MODELS_DIR_V1, "bilstm_v1_best.pth")

SUBMISSION_DATA_PATH = "../../data/raw/submission.csv"
OUTPUT_SUBMISSION_PATH = os.path.join(SUBMISSIONS_DIR, "submission_bilstm_v1_with_O.csv")

os.makedirs(SUBMISSIONS_DIR, exist_ok=True)
print("Информация: Окружение и пути настроены.")

Информация: Окружение и пути настроены.


In [4]:

# --- Класс 1: Модуль для символьных эмбеддингов ---
class CharEmbedding(nn.Module):
    def __init__(self, char_vocab_size, embedding_dim, hidden_dim, dropout_rate=0.25):
        super(CharEmbedding, self).__init__()
        self.embedding = nn.Embedding(char_vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        batch_size, seq_len, word_len = x.size()
        x = x.view(batch_size * seq_len, word_len)
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)
        # Убираем hidden и cell state, так как нам нужен только output
        lstm_out, _ = self.lstm(embedded)
        # Применяем max-pooling по временной оси (длине слова)
        output = lstm_out.permute(0, 2, 1)
        output = torch.max(output, 2)[0]
        output = output.view(batch_size, seq_len, -1)
        return self.dropout(output)

# --- Класс 2: Основная архитектура модели ---
class BiLSTMCrfForNer(nn.Module):
    def __init__(self, word_vocab_size, word_embedding_dim, char_vocab_size, char_embedding_dim, char_hidden_dim, lstm_hidden_dim, num_tags, dropout_rate=0.5, padding_idx=0):
        super(BiLSTMCrfForNer, self).__init__()
        self.word_embedding = nn.Embedding(num_embeddings=word_vocab_size, embedding_dim=word_embedding_dim, padding_idx=padding_idx)
        # Важно: здесь requires_grad должен быть таким же, как при обучении v1. Скорее всего, True.
        self.word_embedding.weight.requires_grad = True

        self.char_embedding = CharEmbedding(char_vocab_size=char_vocab_size, embedding_dim=char_embedding_dim, hidden_dim=char_hidden_dim, dropout_rate=dropout_rate)
        self.embedding_dropout = nn.Dropout(dropout_rate)

        # Убедимся, что dropout в LSTM применяется только если слоев > 1
        lstm_dropout = dropout_rate if 2 > 1 else 0
        self.lstm = nn.LSTM(
            input_size=word_embedding_dim + (2 * char_hidden_dim),
            hidden_size=lstm_hidden_dim,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=lstm_dropout
        )

        self.classifier = nn.Linear(2 * lstm_hidden_dim, num_tags)
        self.crf = CRF(num_tags=num_tags, batch_first=True)

    def forward(self, word_ids, char_ids, mask, tags=None):
        word_embeds = self.word_embedding(word_ids)
        char_embeds = self.char_embedding(char_ids)

        combined_embeds = torch.cat([word_embeds, char_embeds], dim=-1)
        combined_embeds = self.embedding_dropout(combined_embeds)

        lstm_out, _ = self.lstm(combined_embeds)

        emissions = self.classifier(lstm_out)
        mask = mask.bool()

        if self.training and tags is not None:
            # Во время обучения считаем loss
            loss = -self.crf(emissions, tags, mask=mask, reduction='mean')
            return loss
        else:
            # Во время инференса декодируем последовательность
            decoded_tags = self.crf.decode(emissions, mask=mask)
            return decoded_tags

# --- Класс 3: Пайплайн для инференса ---
class NERPipeline:
    def __init__(self, model_path, artefacts_path, tokenizer_name="xlm-roberta-base"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Информация: Пайплайн будет использовать устройство: {self.device}")

        print("Информация: Загрузка артефактов...")
        with open(artefacts_path, "rb") as f:
            artefacts = pickle.load(f)
        self.word2id = artefacts["word2id"]
        self.char2id = artefacts["char2id"]
        self.id2tag = artefacts["id2tag"]
        # Согласованная функция нормализации
        self.tok_norm_fn = lambda t: "".join("0" if c.isdigit() else c for c in t.lower().strip())
        print("Информация: Артефакты успешно загружены.")

        print(f"Информация: Загрузка токенизатора {tokenizer_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

        print("Информация: Инициализация архитектуры модели...")
        self.model = BiLSTMCrfForNer(
            word_vocab_size=len(self.word2id),
            char_vocab_size=len(self.char2id),
            num_tags=len(self.id2tag),
            # Важно: эти гиперпараметры должны точно совпадать с теми, что были при обучении v1
            word_embedding_dim=300,
            char_embedding_dim=50,
            char_hidden_dim=50,
            lstm_hidden_dim=256,
            dropout_rate=0.5,
            padding_idx=self.word2id["<PAD>"]
        )

        print(f"Информация: Загрузка весов модели из {model_path}...")
        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval() # Переводим модель в режим инференса
        print("Информация: Модель готова к работе.")

    def _tokenize_and_get_tokens(self, text: str) -> list[str]:
        """Вспомогательная функция для получения списка токенов без спец. символов."""
        encoding = self.tokenizer(text, add_special_tokens=False)
        tokens = self.tokenizer.convert_ids_to_tokens(encoding["input_ids"])
        return tokens

    def predict(self, text: str) -> list:
        if not text.strip():
            return []

        with torch.no_grad():
            # 1. Токенизация (идентична той, что будет в bio_to_indices)
            tokens = self._tokenize_and_get_tokens(text)
            if not tokens:
                return []

            # 2. Предобработка, полностью идентичная NerDataset
            word_ids = [self.word2id.get(self.tok_norm_fn(token), self.word2id["<UNK>"]) for token in tokens]

            char_ids_per_word = []
            for token in tokens:
                ids = [self.char2id.get(char, self.char2id["<UNK>"]) for char in token]
                char_ids_per_word.append(ids)

            # 3. Паддинг для одного сэмпла (батч размером 1)
            max_word_len = max(len(ids) for ids in char_ids_per_word) if char_ids_per_word else 0
            padded_chars = [ids + [self.char2id["<PAD>"]] * (max_word_len - len(ids)) for ids in char_ids_per_word]

            # 4. Преобразование в тензоры
            word_tensor = torch.tensor([word_ids], dtype=torch.long).to(self.device)
            char_tensor = torch.tensor([padded_chars], dtype=torch.long).to(self.device)
            mask_tensor = torch.tensor([[1] * len(tokens)], dtype=torch.bool).to(self.device)

            # 5. Предсказание
            predictions_ids_list = self.model(word_tensor, char_tensor, mask_tensor)
            if not predictions_ids_list:
                return ['O'] * len(tokens)

            predictions_ids = predictions_ids_list[0]

            # 6. Декодирование
            predicted_tags = [self.id2tag.get(tag_id, 'O') for tag_id in predictions_ids]

            return predicted_tags

print("Информация: Классы для модели и пайплайна определены.")

Информация: Классы для модели и пайплайна определены.


In [5]:
print("--- Инициализация пайплайна с моделью v1 ---")
try:
    pipeline_v1 = NERPipeline(model_path=MODEL_PATH_V1, artefacts_path=ARTEFACTS_PATH_V1)
    print("\nИнформация: Пайплайн успешно инициализирован.")
except Exception as e:
    print(f"Ошибка при инициализации пайплайна: {e}")
    pipeline_v1 = None

--- Инициализация пайплайна с моделью v1 ---
Информация: Пайплайн будет использовать устройство: cpu
Информация: Загрузка артефактов...
Информация: Артефакты успешно загружены.
Информация: Загрузка токенизатора xlm-roberta-base...
Информация: Инициализация архитектуры модели...
Информация: Загрузка весов модели из ../../models/iteration-1/bilstm_v1_best.pth...
Информация: Модель готова к работе.

Информация: Пайплайн успешно инициализирован.


In [6]:
if pipeline_v1:
    print(f"\n--- Генерация предсказаний с помощью модели v1 и нового формата вывода ---")
    submission_df = pd.read_csv(SUBMISSION_DATA_PATH, sep=";")

    all_annotations = []
    for index, row in tqdm(submission_df.iterrows(), total=len(submission_df), desc="Предсказание v1 (с 'O')"):
        text = str(row["sample"])

        predicted_bio_tags = pipeline_v1.predict(text)

        annotations = bio_to_indices(text, predicted_bio_tags)

        all_annotations.append(str(annotations))

    submission_df["annotation"] = all_annotations
    submission_df.to_csv(OUTPUT_SUBMISSION_PATH, sep=";", index=False)

    print("\n--- Процесс завершен ---")
    print(f"Новый файл с предсказаниями сохранен в: {OUTPUT_SUBMISSION_PATH}")
    print("Теперь его можно отправить на проверку для получения 'истинного' бейзлайна.")
else:
    print("Пайплайн не был инициализирован, предсказание невозможно.")


--- Генерация предсказаний с помощью модели v1 и нового формата вывода ---


Предсказание v1 (с 'O'):   0%|          | 0/5000 [00:00<?, ?it/s]


--- Процесс завершен ---
Новый файл с предсказаниями сохранен в: ../../submissions/iteration-1/submission_bilstm_v1_with_O.csv
Теперь его можно отправить на проверку для получения 'истинного' бейзлайна.
